# B1.11 · Remediation engineering

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B1.10 · Exploit chaining](https://spbreed.github.io/cyber-commons/lessons/B1.10.html)**.

| | |
|---|---|
| Tools used | Semgrep OSS, pytest, GLM-4.6, Kimi K2, Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A patch that passes the tests and changes the behaviour is not a fix, it is a second incident with a pull request attached. Remediation is the stage where the pipeline stops finding things and starts touching them.

> **At CyberTravels.** The Coding Agent's fix must not break booking behaviour. A patch that passes the tests and changes what travellers experience is a second incident with a pull request attached. R8.

## 2 · The framework

```
   patch                    what has to be true
   +----------------+       +-----------------------------+
   | fixes the bug  |  and  | behaviour unchanged         |
   |                |       | tests still pass            |
   |                |       | reviewer can follow the why |
   +----------------+       +-----------------------------+

   a patch that passes the tests and changes the behaviour is
   a second incident with a pull request attached
```

**Stage 14 — Remediation engineering.** Generate the fix, then prove it.

A model that finds bugs is useful. A model that fixes them is only useful if you
can tell a real fix from a plausible one, and plausible is exactly what language
models are optimised to produce.

There are three ways to make a finding stop firing:

1. **Fix the vulnerability** — behaviour preserved, bug gone.
2. **Remove the code** — finding gone, so is the feature.
3. **Evade the detector** — rewrite until the pattern misses.

All three make the scanner green, and an autonomous loop optimising for a green
scan will find options 2 and 3 on its own because they are cheaper.

The pipeline has an advantage a static workflow does not: Phase 4 already built
a working exploit. So the acceptance test is not "does the scanner still fire?"
It is **"does the exploit still work against the patched build?"** — which is
the only question that cannot be gamed by editing the code around the detector.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · The model backend, and the patch it proposes

In [ ]:
# --- model backend: replay by default, real model when you configure one ----
# Nothing here is Anthropic- or vendor-specific beyond one URL and one header
# shape. Standard library only, so the notebook stays self-contained.
import json, os, urllib.error, urllib.request

# The cheapest current model on each side, which is what a lesson needs.
FRONTIER_DEFAULT   = "claude-haiku-4-5-20251001"
OPEN_WEIGHT_DEFAULT = "glm-4.6"
TIMEOUT = 60

def _kaggle_secret(name):
    """On Kaggle, a key lives in Add-ons -> Secrets rather than the environment.

    kaggle_secrets is pre-installed in the Kaggle image and absent everywhere
    else, so the import is guarded and the notebook needs no dependency. It also
    requires the notebook to have internet enabled, which on Kaggle requires a
    phone-verified account - see the note printed below.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("ANTHROPIC_API_KEY") or _kaggle_secret("ANTHROPIC_API_KEY"):
        os.environ.setdefault("ANTHROPIC_API_KEY",
                              os.environ.get("ANTHROPIC_API_KEY")
                              or _kaggle_secret("ANTHROPIC_API_KEY") or "")
        return "frontier", os.environ.get("MODEL", FRONTIER_DEFAULT)
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _anthropic(prompt, system, model, max_tokens, temperature):
    body = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    out = _post("https://api.anthropic.com/v1/messages", body,
                {"x-api-key": os.environ["ANTHROPIC_API_KEY"],
                 "anthropic-version": "2023-06-01"})
    return "".join(b.get("text", "") for b in out.get("content", [])).strip()

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        fn = _anthropic if kind == "frontier" else _openai_compatible
        return fn(prompt, system, model, max_tokens, temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        detail = getattr(e, "code", None) or type(e).__name__
        print(f"   !! {kind} backend ({model}) failed: {detail} - using the replay,")
        print("      which is a replay and is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, set one of:")
    print()
    print("   frontier     export ANTHROPIC_API_KEY=...   # cheapest: " + FRONTIER_DEFAULT)
    print("   open weight  export OPENAI_BASE_URL=http://localhost:11434/v1 \\")
    print("                       OPENAI_API_KEY=ollama MODEL=glm-4.6")
    print()
    print("   On Kaggle: Add-ons -> Secrets, add ANTHROPIC_API_KEY, and switch")
    print("   Internet on in the notebook settings. Internet requires a")
    print("   phone-verified Kaggle account; without it DNS fails in the kernel")
    print("   and this lesson correctly stays on the replay.")

## 4 · The same lesson, against a real model

Everything below this point runs identically on three backends. Offline it uses
a deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `ANTHROPIC_API_KEY` set it calls a frontier
model; with `OPENAI_BASE_URL` set it calls any OpenAI-compatible endpoint,
which covers Ollama, vLLM and the hosted open-weight providers.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'Fix this without changing the function\'s behaviour for valid input. Return only the patched function.\n\ndef report(request):\n    q = "SELECT * FROM orders WHERE ref = \'" + request.args[\'ref\'] + "\'"\n    return db.execute(q)'

REPLAY = 'def report(request):\n    q = "SELECT * FROM orders WHERE ref = ?"\n    return db.execute(q, (request.args[\'ref\'],))'

answer, used, model = ask(TASK, replay=REPLAY,
            system='You are a remediation engineer. Output code only, no explanation.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("parameterises the query", "?" in answer or "%s" in answer or ":ref" in answer)
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, three possible backends. Offline the answer")
print("is the replay and is labelled as one; with a key it is the model's.")

## 5 · The confirmed finding, with its working exploit

In [ ]:
import re, sqlite3

VULNERABLE = '''
def get_user(conn, name):
    return conn.execute("SELECT id, name FROM users WHERE name = '" + name + "'").fetchall()
'''

def build_db():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE users(id INTEGER, name TEXT)")
    conn.executemany("INSERT INTO users VALUES (?,?)",
                     [(1,"dana"),(2,"sam"),(3,"o'brien")])
    return conn

def load(src):
    ns = {}; exec(compile(src, "<patch>", "exec"), ns); return ns["get_user"]

BEHAVIOUR = [("dana",[(1,"dana")]), ("sam",[(2,"sam")]),
             ("nobody",[]), ("o'brien",[(3,"o'brien")])]

def behaviour_ok(fn):
    conn = build_db(); rows = []
    for name, expected in BEHAVIOUR:
        try: got = fn(conn, name)
        except Exception as e: rows.append((name, f"raised {type(e).__name__}", False)); continue
        rows.append((name, got, got == expected))
    return rows

def exploit_works(fn):
    """The stage-12 probe, reused as the acceptance test."""
    conn = build_db()
    try: rows = fn(conn, "x' OR '1'='1")
    except Exception: return False, "probe raised — not exploitable this way"
    return len(rows) > 1, f"probe returned {len(rows)} rows"

def scanner_fires(src):
    return bool(re.search(r"execute\(\s*[\"\'][^\"\']*[\"\']\s*\+", src))

fn = load(VULNERABLE)
print("behaviour of the vulnerable build:")
for name, got, ok in behaviour_ok(fn):
    print(f"   get_user({name!r:10s}) → {str(got):18s} {'ok' if ok else 'FAILS'}")
ex, why = exploit_works(fn)
print(f"\nexploit works: {ex} — {why}")
print(f"scanner fires: {scanner_fires(VULNERABLE)}")

## 6 · Four candidate patches, three of which make CI green

In [ ]:
CANDIDATES = {
 "A · parameterise (the real fix)": '''
def get_user(conn, name):
    return conn.execute("SELECT id, name FROM users WHERE name = ?", (name,)).fetchall()
''',
 "B · delete the feature": '''
def get_user(conn, name):
    return []
''',
 "C · evade the scanner": '''
def get_user(conn, name):
    q = "SELECT id, name FROM users WHERE name = '%s'" % name
    return conn.execute(q).fetchall()
''',
 "D · escape by hand": '''
def get_user(conn, name):
    safe = name.replace("'", "''")
    return conn.execute("SELECT id, name FROM users WHERE name = '" + safe + "'").fetchall()
''',
}
print(f"{'candidate':34s}{'scanner green':>15}")
print("-" * 50)
for name, src in CANDIDATES.items():
    print(f"{name:34s}{str(not scanner_fires(src)):>15}")
print("\nThree of four are green. Only one of those is a fix.")

## 7 · The control — validate on three axes, exploit first

In [ ]:
def validate(src):
    fn = load(src)
    green = not scanner_fires(src)
    beh = behaviour_ok(fn)
    preserved = all(ok for _, _, ok in beh)
    still_exploitable, _ = exploit_works(fn)
    reasons = []
    if not green:            reasons.append("scanner still fires")
    if not preserved:        reasons.append("behaviour changed")
    if still_exploitable:    reasons.append("STILL EXPLOITABLE (stage-12 probe passes)")
    return (not reasons), green, preserved, still_exploitable, reasons

print(f"{'candidate':34s}{'scan':6s}{'behaviour':11s}{'exploitable':13s}verdict")
print("-" * 84)
accepted = []
for name, src in CANDIDATES.items():
    ok, g, b, x, reasons = validate(src)
    if ok: accepted.append(name)
    print(f"{name:34s}{str(g):6s}{str(b):11s}{str(x):13s}"
          f"{'ACCEPT' if ok else 'REJECT — ' + ', '.join(reasons)}")
print(f"\naccepted: {accepted}")
assert "A · parameterise (the real fix)" in accepted
assert "B · delete the feature" not in accepted
assert "C · evade the scanner" not in accepted

In [ ]:
# The proof-of-fix clause: the exploit must fail on the new build and
# succeed on the old one. Without both halves, "fixed" is a claim.
def proof_of_fix(old_src, new_src):
    old_ex, _ = exploit_works(load(old_src))
    new_ex, _ = exploit_works(load(new_src))
    return (old_ex and not new_ex), f"exploit on old={old_ex}, on new={new_ex}"

for name in accepted:
    ok, detail = proof_of_fix(VULNERABLE, CANDIDATES[name])
    print(f"{name:34s} proof of fix: {ok}  ({detail})")

print("\nCandidate D passes every automated check and is still the wrong answer:")
print("it reimplements the driver's escaping and will be wrong for the next")
print("input class or the next database. Nothing except a rule about MECHANISM")
print("catches that — which is the part of remediation that does not automate.")

## What you just proved

The vulnerable build passes all four behaviour cases and the exploit returns 3 rows. Candidates A, B and D make the scanner green. Validation rejects B for changed behaviour and C for remaining exploitable, accepting A and D. Proof of fix holds for both accepted patches — the exploit works on the old build and fails on the new.

## Your turn

Candidate D passes every automated gate and is still wrong. Write the rule that rejects it. You will find it has to be about which *mechanism* is acceptable, not about outcomes — and that rule belongs in your secure coding standard, not in the pipeline.

---

**Next → [B1.12 · Severity calibration and reporting](https://spbreed.github.io/cyber-commons/lessons/B1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*